# LLMs — Arquitectura, Ecosistema y Prompting

### UTN-FRSF · Ingeniería en Sistemas de Información · Inteligencia Artificial
**Dr. Jorge Roa · Dra. María de los Milagros Gutiérrez**

---

*¿Cómo funciona un LLM y cómo se le habla bien?*

## Recap Clase 1 → conexión con hoy

En Clase 1 construimos los cimientos:

```
  Texto     ──▶  Tokenización  ──▶  Embeddings  ──▶  ???  ──▶  Output
  (crudo)         (subpalabras)      (vectores)      (hoy)
```

Hoy abrimos la caja del medio:

| Lo que ya sabemos | Lo que vemos hoy |
|---|---|
| Tokenizers: BPE, WordPiece, SentencePiece | Esos mismos en GPT-4o, Llama, BERT |
| Embeddings: vectores de 384 dimensiones | Cómo el Transformer los procesa |
| BoW / TF-IDF: representación ingenua | Por qué attention lo supera |

> Los embeddings del corpus de Clase 1 los volvemos a usar en Clase 3 (RAG).

## ⚙️ Setup

```bash
pip install transformers tiktoken torch bertviz
pip install groq ollama openai   # para B5 — prompting
```

**Modelos que usamos hoy:**
- `tiktoken` — tokenizer GPT-4o, local sin API key
- `bert-base-multilingual-cased` — via HuggingFace, ~680MB
- `Qwen/Qwen2.5-0.5B` — tokenizer Qwen (opcional), ~1MB solo tokenizer
- `BertViz` — visualización de attention weights
- Groq API o Ollama — para ejercicios de prompting

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import tiktoken
from transformers import AutoTokenizer
import pandas as pd

# Frase de prueba — la usamos en toda la clase
FRASE_ES = "La inteligencia artificial está transformando la ingeniería de software."
FRASE_EN = "Artificial intelligence is transforming software engineering."

print('✓ Setup OK')
print(f'  Frase ES: "{FRASE_ES}"')
print(f'  Frase EN: "{FRASE_EN}"')

# Bloque 1
## Tokenizers en modelos reales

---

*Ya saben qué son BPE y WordPiece. Ahora los vemos en los modelos que van a usar en el curso.*

## Los mismos algoritmos, distintos modelos

| Modelo | Algoritmo | Vocab | Librería |
|---|---|---|---|
| **GPT-4o** | BPE `o200k_base` | ~200K | `tiktoken` — local |
| **Llama 3.1** | BPE `cl100k_base` | ~128K | `tiktoken` — local |
| **BERT multilingual** | WordPiece | ~120K | `transformers` — local |
| **Qwen2.5 / Qwen3** | BPE (tiktoken-based) | ~150K | `tiktoken` / `transformers` |

El algoritmo es el mismo de Clase 1 — la diferencia está en los vocabularios entrenados con billones de tokens.

> **Recordar:** el tokenizer corre 100% local, sin costo, sin API key.

> 📖 *Sennrich, R., et al. (2016). Subword Units. ACL 2016. https://arxiv.org/abs/1508.07909*  
> 📖 *Devlin, J., et al. (2018). BERT. NAACL 2019. https://arxiv.org/abs/1810.04805*

In [ ]:
# Comparación de tokenizers sobre la misma frase
# tiktoken tiene encodings para GPT-4o y Llama 3 sin necesidad de API key

enc_gpt4o  = tiktoken.get_encoding('o200k_base')   # GPT-4o
enc_llama3 = tiktoken.get_encoding('cl100k_base')  # Llama 3 / GPT-4

tok_bert = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

resultados = []
for frase, label in [(FRASE_ES, 'ES'), (FRASE_EN, 'EN')]:
    t_gpt4o  = enc_gpt4o.encode(frase)
    t_llama3 = enc_llama3.encode(frase)
    t_bert   = tok_bert.tokenize(frase)
    resultados.append({
        'Frase': label,
        'GPT-4o (BPE o200k)': len(t_gpt4o),
        'Llama 3 (BPE cl100k)': len(t_llama3),
        'BERT (WordPiece)': len(t_bert),
    })

df = pd.DataFrame(resultados).set_index('Frase')
print('Cantidad de tokens por modelo y frase:')
print(df.to_string())

In [ ]:
# Ver cómo parte cada tokenizer las mismas palabras
print('═' * 60)
print('GPT-4o (BPE o200k_base):')
tokens_gpt = enc_gpt4o.encode(FRASE_ES)
print([enc_gpt4o.decode([t]) for t in tokens_gpt])

print()
print('BERT multilingual (WordPiece):')
print(tok_bert.tokenize(FRASE_ES))

print()
print('Observar las diferencias:')
print('  GPT-4o: parte por subpalabras según frecuencia en corpus GPT')
print('  BERT:   usa ## para marcar continuación de palabra')
print('  Ambos usan el mismo concepto de Clase 1, con vocabularios distintos')

In [ ]:
# OPCIONAL: Qwen tokenizer (requiere HuggingFace)
# Qwen2.5 y Qwen3 usan un tokenizer BPE propio derivado de tiktoken

try:
    tok_qwen = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B', trust_remote_code=True)
    tokens_qwen = tok_qwen.tokenize(FRASE_ES)
    print(f'Qwen2.5 (BPE, vocab ~150k):')
    print(tokens_qwen)
    print(f'  → {len(tokens_qwen)} tokens para la frase en español')
except Exception as e:
    print(f'Qwen no disponible localmente: {e}')
    print('Podés acceder vía: pip install transformers && huggingface-cli login')

## Tokens y costo — lo que nadie te dice

Los modelos via API cobran por token. Precios aproximados a abril 2026:

| Modelo | Input (por 1M tokens) | Output (por 1M tokens) |
|---|---|---|
| GPT-5.4 | ~\$15 | ~\$60 |
| Claude Opus 4.6 | ~\$15 | ~\$75 |
| Claude Sonnet 4.6 | ~\$3 | ~\$15 |
| Gemini 3.1 Pro | ~\$3.5 | ~\$10 |
| **Qwen3 8B (Groq)** | **~\$0.10** | **~\$0.10** |
| **Ollama local** | **\$0** | **\$0** |

💡 **Consecuencia práctica:**  
- Escribir en español puede costar 30-50% más que en inglés (más tokens)  
- Para desarrollo y testing → Ollama local o Groq free tier  
- Para producción → elegir según el trade-off calidad/costo del caso de uso

# Bloque 2
## Arquitectura Transformer

---

*Antes: RNN/LSTM procesaban de izquierda a derecha, secuencialmente.  
Problema: al llegar a la palabra 50, ya "olvidabas" la palabra 1.*

*Transformer lo resuelve: procesa todo en paralelo.*

## Attention — la intuición central

### *"el banco quebró porque estaba podrido"*

Para entender qué significa **"banco"** en esta oración, el modelo mira **todas** las otras palabras y decide cuáles son relevantes:

```
  banco  ←─────────────────────── podrido  (muy relevante: madera → mueble)
  banco  ←───────────── quebró            (relevante: puede ser financiero)
  banco  ←──── el                         (poco relevante)
```

La palabra **"podrido"** tiene alto peso de atención → `banco = mueble`, no institución.

Esto resuelve la **polisemia** que BoW y Word2Vec no podían manejar.

> 📖 *Vaswani, A., et al. (2017). Attention Is All You Need. NeurIPS 2017. https://arxiv.org/abs/1706.03762*

## Attention — Q, K, V sin matemática pesada

```
  Para cada token, el modelo genera tres vectores:

  Q (Query)  = "¿qué estoy buscando?"
  K (Key)    = "¿qué información tengo?"
  V (Value)  = "¿qué voy a devolver si me buscan?"

  ┌──────────┐     ┌──────────┐     ┌──────────┐
  │  Query   │     │   Keys   │     │  Values  │
  │ "banco"  │ ──▶ │ todas las│ ──▶ │ combino  │
  │ ¿qué soy?│     │ palabras │     │ según    │
  └──────────┘     └──────────┘     │ similitud│
                                     └──────────┘
                                          │
                                          ▼
                                   representación
                                   enriquecida de
                                   "banco" según
                                   el contexto
```

El modelo hace esto para **todos los tokens a la vez** (en paralelo) y con **múltiples cabezas de atención** simultáneas.

## Encoder vs Decoder vs Encoder-Decoder

| Arquitectura | Para qué sirve | Ejemplos | ¿Mira el futuro? |
|---|---|---|---|
| **Solo Encoder** | Entender texto (clasificación, embeddings) | BERT, RoBERTa | ✓ Bidireccional |
| **Solo Decoder** | Generar texto (siguiente token) | GPT, Llama, Qwen, Claude | ✗ Solo el pasado |
| **Encoder-Decoder** | Transformar texto (traducción, resumen) | T5, BART, mT5 | Mixto |

```
  BERT (Encoder):                    GPT (Decoder):
  "el banco [MASK] dinero"           "el banco"
        ↕↕↕↕↕↕↕↕                          ↓
  ve TODO el contexto                 "quebró"  (solo ve lo anterior)
  → predice [MASK] = "tiene"               ↓
                                      "porque"
                                            ↓
                                       "estaba"
```

> En este curso usamos modelos **Decoder** (GPT, Llama, Qwen) — generan texto.

## Escala — por qué importa

```
  Parámetro = un número que el modelo aprendió durante el entrenamiento

  GPT-2    (2019):   1.5 B parámetros   → escribe texto coherente
  GPT-3    (2020):   175 B parámetros   → razona, traduce, programa
  Llama 3  (2024):   8B / 70B / 405B    → comparable a GPT-4
  Qwen3    (2025):   0.6B → 235B        → familia completa
  Kimi K2.5(2026):   1T total / 32B activos (MoE)
```

**¿Por qué MoE (Mixture of Experts)?**
```
  Modelo denso:   activa TODOS los parámetros para cada token  → costoso
  Modelo MoE:     activa solo los "expertos" relevantes         → eficiente

  Kimi K2.5: 1T parámetros totales, pero solo 32B activos por request
  Resultado: calidad de modelo grande, costo de modelo pequeño
```

## 🔽 Bonus: ¿Cómo funciona MoE internamente?

```
  Modelo DENSO (ej: Llama 3 8B):
  ─────────────────────────────────────────────
  Input Token  →  [TODAS las 8B neuronas activas]  →  Output
                  (siempre, sin excepción)
  Costo: proporcional al tamaño total del modelo


  Modelo MoE (ej: Kimi K2.5, 1T params / 32B activos):
  ─────────────────────────────────────────────────────
                    ┌─────────────┐
  Input Token  →   │   ROUTER    │   (aprende a decidir)
                    └──────┬──────┘
                           │
          ┌────────────────┼──────────────────┐
          ▼                ▼                  ▼
    [Experto 1]      [Experto 4]        [Experto 11]    ← solo 3 de 384 expertos
    (código)         (matemática)       (razonamiento)
          │                │                  │
          └────────────────┴──────────────────┘
                           │
                        Output

  Costo: proporcional a los expertos ACTIVOS (32B), no al total (1T)
```

**Por qué importa para nosotros:**
- Podemos acceder a calidad de modelo grande pagando inferencia de modelo pequeño
- Groq, OpenRouter y otros proveedores pueden servir modelos MoE de forma muy económica
- Kimi K2.5 via API: ~$0.60/M input tokens (comparable a Llama 7B denso)

> 📖 *Shazeer, N., et al. (2017). Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer. ICLR 2017. https://arxiv.org/abs/1701.06538*

In [ ]:
# Visualización de attention weights con BertViz
# Permite ver qué palabras "se miran" entre sí durante el attention

try:
    from bertviz import head_view
    from transformers import BertTokenizer, BertModel
    import torch

    model_name = 'bert-base-multilingual-cased'
    tokenizer_bert = BertTokenizer.from_pretrained(model_name)
    model_bert = BertModel.from_pretrained(model_name, output_attentions=True)
    model_bert.eval()

    oracion = "el banco quebró porque estaba podrido"
    inputs = tokenizer_bert.encode_plus(oracion, return_tensors='pt', add_special_tokens=True)

    with torch.no_grad():
        outputs = model_bert(**inputs)

    attention      = outputs.attentions
    tokens_display = tokenizer_bert.convert_ids_to_tokens(inputs['input_ids'][0])

    print(f'Tokens: {tokens_display}')
    print(f'Capas de attention: {len(attention)}')
    print(f'Cabezas por capa:   {attention[0].shape[1]}')
    print()
    print('Ejecutar en Jupyter para ver la visualización interactiva:')
    print('  head_view(attention, tokens_display)')

    # En Jupyter: descomentar la línea siguiente
    # head_view(attention, tokens_display)

except ImportError:
    print('bertviz no instalado. Instalar con: pip install bertviz')
    print('Luego ejecutar en Jupyter para ver la visualización interactiva.')

In [ ]:
# Si no podés instalar BertViz: visualización manual de attention weights
# Muestra qué tokens se "prestan atención" entre sí

try:
    import torch
    from transformers import BertTokenizer, BertModel

    tokenizer_bert = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model_bert = BertModel.from_pretrained('bert-base-multilingual-cased', output_attentions=True)
    model_bert.eval()

    oracion = "el banco quebró porque estaba podrido"
    inputs  = tokenizer_bert.encode_plus(oracion, return_tensors='pt')
    tokens  = tokenizer_bert.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        outputs = model_bert(**inputs)

    # Capa 0, cabeza 0 — attention desde el token "banco"
    attn       = outputs.attentions[0][0, 0].numpy()  # (seq, seq)
    idx_banco  = tokens.index('banco') if 'banco' in tokens else 2

    print(f'Attention desde "{tokens[idx_banco]}" hacia cada token:')
    print('-' * 45)
    for tok, score in zip(tokens, attn[idx_banco]):
        barra = '█' * int(score * 40)
        print(f'  {tok:<15} {score:.4f}  {barra}')

    print()
    print('→ Los tokens con barra más larga son los que más influyen en la representación de "banco"')

except Exception as e:
    print(f'Transformers no disponible: {e}')
    print('Instalar con: pip install transformers torch')

In [ ]:
# 🔽 Bonus: Heatmap de attention — visualización sin BertViz
# Muestra TODAS las cabezas de atención de una capa en una grilla

try:
    import torch
    import matplotlib.pyplot as plt
    import numpy as np
    from transformers import BertTokenizer, BertModel

    tokenizer_bert = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model_bert = BertModel.from_pretrained('bert-base-multilingual-cased', output_attentions=True)
    model_bert.eval()

    oracion = "el banco quebró porque estaba podrido"
    inputs  = tokenizer_bert.encode_plus(oracion, return_tensors='pt')
    tokens  = tokenizer_bert.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        outputs = model_bert(**inputs)

    # Capa 0: 12 cabezas de atención
    attn_layer0 = outputs.attentions[0][0].numpy()  # (12 heads, seq, seq)
    n_heads = min(6, attn_layer0.shape[0])

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    fig.suptitle(f'Attention weights — BERT Capa 0 (primeras {n_heads} cabezas)
"{oracion}"',
                 fontsize=11)

    for h, ax in enumerate(axes.flat):
        if h >= n_heads:
            ax.axis('off'); continue
        im = ax.imshow(attn_layer0[h], cmap='Blues', vmin=0, vmax=attn_layer0[h].max())
        ax.set_title(f'Cabeza {h+1}', fontsize=9)
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(tokens)))
        ax.set_yticklabels(tokens, fontsize=7)

    plt.colorbar(im, ax=axes.flat[-1], shrink=0.8)
    plt.tight_layout()
    plt.savefig('attention_heatmap.png', dpi=130, bbox_inches='tight')
    plt.show()

    print('→ Cada celda (i,j) = cuánto el token i presta atención al token j')
    print('→ Diferentes cabezas capturan diferentes tipos de relaciones')
    print('   (sintácticas, semánticas, co-referencia, etc.)')

except Exception as e:
    print(f'Error: {e}')
    print('Requiere: pip install transformers torch matplotlib')

## 🔽 Bonus: La fórmula completa de Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V$$

### Desglose paso a paso:

**1. `Q · Kᵀ` — similitud entre query y todas las keys**
```
  Para cada token i, calculo qué tan similar es su Query
  con la Key de cada otro token j.
  Resultado: matriz (seq_len × seq_len) de scores
```

**2. `/ √d_k` — escalado**
```
  Dividir por √d_k (raíz de la dimensión de los vectores)
  evita que los dot products sean muy grandes cuando d_k es alto.
  Sin esto, softmax se satura → gradientes muy pequeños → aprendizaje lento.
```

**3. `softmax(...)` — normalización**
```
  Convierte los scores en probabilidades (suman 1).
  Token "podrido" → score alto respecto a "banco" → peso alto.
  Token "el"      → score bajo respecto a "banco" → peso bajo.
```

**4. `· V` — combinación ponderada**
```
  El output para "banco" es una suma ponderada de los Values de todos los tokens.
  → La representación de "banco" absorbe información de "podrido".
  → Polisemia resuelta por contexto.
```

> 📖 *Vaswani, A., et al. (2017). Attention Is All You Need. NeurIPS 2017. https://arxiv.org/abs/1706.03762*  
> 📖 *Jurafsky, D. & Martin, J. H. (2024). SLP3, Cap. 9. https://web.stanford.edu/~jurafsky/slp3/9.pdf*

# Bloque 3
## Ciclo de vida del modelo

---

*¿Cómo pasa un modelo de "completar texto" a "seguir instrucciones"?*

## Los tres estadios de entrenamiento

```
  1. PRE-TRAINING
  ───────────────
  Datos:    Billones de tokens de internet, libros, código
  Objetivo: Predecir el siguiente token
  Resultado: El modelo "sabe el idioma" pero no sabe qué hacer con él

  Ejemplo:  "La capital de Francia es" → completa con "París, y también..."


  2. SFT — Supervised Fine-Tuning
  ────────────────────────────────
  Datos:    Miles de pares (pregunta → respuesta ideal) curados por humanos
  Objetivo: Aprender a seguir instrucciones
  Resultado: El modelo responde preguntas en lugar de completar texto

  Ejemplo:  "¿Cuál es la capital de Francia?" → "París."


  3. RLHF / DPO — Alineación con preferencias humanas
  ─────────────────────────────────────────────────────
  Datos:    Humanos ranquean respuestas alternativas (A mejor que B)
  Objetivo: Aprender qué respuestas prefieren los humanos
  Resultado: El modelo responde de forma más útil, segura y honesta

  RLHF: usa un modelo de recompensa + RL
  DPO:  más simple, optimiza directamente sobre los pares preferidos
```

> 📖 *Ouyang, L., et al. (2022). Training language models to follow instructions with human feedback. NeurIPS 2022. https://arxiv.org/abs/2203.02155*  
> 📖 *Rafailov, R., et al. (2023). Direct Preference Optimization. NeurIPS 2023. https://arxiv.org/abs/2305.18290*

## Base model vs Instruct model

La consecuencia práctica del ciclo de vida:

```
  MISMO PROMPT: "La capital de Francia es"

  Base model (solo pre-training):
  ──────────────────────────────
  "París, y también es conocida como la Ciudad de la Luz.
   La capital de Alemania es Berlín, y la de Italia es Roma..."
  → Sigue completando texto, no se detiene

  Instruct model (pre-training + SFT + RLHF):
  ────────────────────────────────────────────
  "París."
  → Entiende que es una pregunta implícita y responde concisamente
```

**Nomenclatura:**
- `Qwen3-8B` → base model  
- `Qwen3-8B-Instruct` → instruct model ← el que usamos
- `Llama-3.1-8B` → base
- `Llama-3.1-8B-Instruct` → instruct ← el que usamos

In [ ]:
# Demo: diferencia entre base model e instruct model via Ollama
# Requiere: ollama pull qwen3:8b

try:
    import ollama

    prompt = "La capital de Francia es"

    print('Probando con qwen3:8b (instruct)...')
    print(f'Prompt: "{prompt}"')
    print()

    resp = ollama.generate(
        model='qwen3:8b',
        prompt=prompt,
        options={'num_predict': 60, 'temperature': 0.1}
    )
    print('Respuesta (modelo instruct):')
    print(f'  {resp["response"].strip()}')
    print()
    print('→ Un base model completaría el texto. El instruct entiende que es una pregunta.')

except Exception as e:
    print(f'Ollama no disponible: {e}')
    print('Instalar con: https://ollama.com  →  ollama pull qwen3:8b')

# Bloque 4
## Ecosistema actual — abril 2026

---

*¿Qué modelos existen hoy y cómo elegir?*

## Modelos closed source — los top del momento

| Modelo | Organización | Fortaleza principal | Contexto |
|---|---|---|---|
| **GPT-5.4** | OpenAI | General, reasoning, multimodal | 200K tokens |
| **Claude Opus 4.6** | Anthropic | Coding, escritura, reasoning | 200K tokens |
| **Claude Sonnet 4.6** | Anthropic | Balance calidad/costo | 200K tokens |
| **Gemini 3.1 Pro** | Google | Multimodal, contexto gigante | 1M tokens |
| **Grok 4** | xAI | Acceso a datos en tiempo real | 1M tokens |

**Cómo elegir:** no existe un modelo "mejor" en todo.
- Coding → Claude Opus 4.6
- Contexto muy largo → Gemini 3.1 Pro
- Balance costo/calidad → Claude Sonnet 4.6 o GPT-5.4
- Gratis para desarrollo → Groq API (Llama/Qwen) o Ollama local

## Modelos open weight — lo más relevante hoy

| Modelo | Org | Params activos | Local | Destaca en |
|---|---|---|---|---|
| **Llama 4 Scout** | Meta | 17B (MoE 109B) | parcial | Contexto 10M tokens |
| **Qwen3 / Qwen3.5** | Alibaba | 4B–72B | ✓ Ollama | Multilingüe, liviano |
| **Kimi K2.5** | Moonshot | 32B (MoE 1T) | ✗ muy grande | Agentes, coding |
| **Mistral Small 4** | Mistral | 22B | ✓ Ollama | Eficiente |
| **Gemma 4** | Google | 4B–27B | ✓ Ollama | On-device |
| **DeepSeek-R1** | DeepSeek | varios | parcial | Razonamiento |

**En este curso:** `qwen3:8b` o `qwen3:4b` via Ollama — gratuito, local, Apache 2.0.

## Kimi K2.5 — el modelo de la semana

Lanzado en enero 2026 por **Moonshot AI** (China). Open weight.

```
  Arquitectura:  MoE — 1 trillón de parámetros totales
                        32B parámetros activos por request
  Entrenamiento: 15 trillones de tokens visuales + texto
  Modos:         Instant · Thinking · Agent · Agent Swarm

  Agent Swarm:
  ┌─────────────────────────────────────────────────────┐
  │  Orquestador (Kimi K2.5)                            │
  │       │                                             │
  │  ┌────┴────────────────────────────────────────┐   │
  │  │                                             │   │
  │  ▼         ▼         ▼         ▼         ▼     │   │
  │ Agent1   Agent2   Agent3   Agent4   Agent5 ... │   │
  │ (hasta 100 agentes ejecutando en paralelo)     │   │
  └────────────────────────────────────────────────┘   │
  └─────────────────────────────────────────────────────┘
```

Por qué lo mencionamos: **es el tipo de capacidad agentica que vemos en Clase 4**,  
ahora disponible en un modelo open weight.

## Benchmarks — qué miden y qué NO miden

**Lo que sí miden:**
- MMLU: conocimiento general (exámenes universitarios)
- HumanEval / SWE-Bench: generación de código
- MATH: razonamiento matemático
- HLE (Humanity's Last Exam): preguntas muy difíciles de ciencia

**Lo que NO miden:**
```
  ✗ Utilidad real en tu caso de uso específico
  ✗ Consistencia en conversaciones largas
  ✗ Manejo de idiomas distintos al inglés
  ✗ Comportamiento con datos privados de tu empresa
  ✗ Costo de operación en producción
  ✗ Latencia en producción con carga real
```

> **Regla práctica:** los benchmarks son para descartar modelos malos, no para elegir el mejor.  
> El mejor modelo es el que funciona en tu caso de uso con tu presupuesto.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Mapa cualitativo: calidad vs costo
modelos = {
    # (costo relativo, calidad relativa, open_weight, label)
    'GPT-5.4':          (9.5, 9.5, False),
    'Claude Opus 4.6':  (9.0, 9.2, False),
    'Gemini 3.1 Pro':   (7.5, 9.0, False),
    'Claude Sonnet 4.6':(5.0, 8.5, False),
    'Kimi K2.5':        (3.0, 8.8, True),
    'Llama 4 Scout':    (2.0, 8.2, True),
    'Qwen3 8B':         (0.5, 7.5, True),
    'Mistral Small 4':  (1.5, 7.8, True),
    'Gemma 4 9B':       (0.5, 7.2, True),
    'Ollama local':     (0.0, 7.0, True),
}

fig, ax = plt.subplots(figsize=(10, 6))

for nombre, (costo, calidad, open_w) in modelos.items():
    color  = '#1D9E75' if open_w else '#7F77DD'
    marker = 'o' if open_w else 's'
    ax.scatter(costo, calidad, c=color, marker=marker, s=120, zorder=3)
    ax.annotate(nombre, (costo, calidad), textcoords='offset points',
                xytext=(6, 4), fontsize=8.5)

ax.set_xlabel('Costo relativo (0=gratis, 10=muy caro)', fontsize=11)
ax.set_ylabel('Calidad relativa', fontsize=11)
ax.set_title('Ecosistema LLM — Abril 2026 (referencial)', fontsize=12)
ax.grid(True, alpha=0.25)
ax.set_xlim(-0.5, 11); ax.set_ylim(6.5, 10)
ax.spines[['top','right']].set_visible(False)

legend = [
    mpatches.Patch(color='#1D9E75', label='Open weight'),
    mpatches.Patch(color='#7F77DD', label='Closed source'),
]
ax.legend(handles=legend, loc='lower right')
plt.tight_layout()
plt.savefig('ecosistema_llm_2026.png', dpi=150, bbox_inches='tight')
plt.show()

# Bloque 5
## Prompting como ingeniería

---

*El prompt es la única interfaz que tenés con el modelo. Usarla bien hace toda la diferencia.*

## System prompt — la capa de configuración

```
  Estructura de una llamada a un LLM:

  ┌─────────────────────────────────────────────────────┐
  │  SYSTEM (invisible para el usuario)                 │
  │  "Sos un asistente experto en derecho argentino.    │
  │   Respondés en español formal. Siempre citás        │
  │   el artículo del código civil correspondiente."    │
  ├─────────────────────────────────────────────────────┤
  │  USER                                               │
  │  "¿Qué pasa si no pago el alquiler?"                │
  ├─────────────────────────────────────────────────────┤
  │  ASSISTANT                                          │
  │  "Según el Código Civil y Comercial, Art. 1222..."  │
  └─────────────────────────────────────────────────────┘
```

El system prompt define:
- **Rol:** quién es el modelo en este contexto
- **Restricciones:** qué puede y qué no puede hacer
- **Formato:** cómo debe responder
- **Contexto:** información que necesita para operar

## Las 4 técnicas fundamentales

| Técnica | Cuándo usarla | Costo de tokens |
|---|---|---|
| **Zero-shot** | Tarea simple, modelo capaz | Mínimo |
| **Few-shot** | Formato específico, tarea nueva | Medio |
| **Chain of Thought** | Razonamiento, matemática, lógica | Alto |
| **Structured output** | Integración con código, JSON | Medio |

```
  Zero-shot:  "Clasificá este email: [email]"  →  spam / no spam

  Few-shot:   "Email 1: [texto] → spam
               Email 2: [texto] → no spam
               Email 3: [texto] →"               ← el modelo completa

  CoT:        "Pensá paso a paso antes de responder.
               ¿Cuántos tokens tiene este texto?"

  Structured: "Respondé SOLO en JSON con campos: {spam: bool, razon: str}"
```

In [ ]:
# Setup para ejercicios de prompting
# Opción A: Ollama local (sin costo, requiere ollama instalado)
# Opción B: Groq API (gratis con límites, requiere API key)

USE_OLLAMA = True  # cambiar a False para usar Groq

def llamar_llm(messages, model=None, temperature=0.7):
    """Wrapper unificado para Ollama o Groq."""
    if USE_OLLAMA:
        import ollama
        model = model or 'qwen3:8b'
        resp  = ollama.chat(model=model, messages=messages,
                            options={'temperature': temperature})
        return resp['message']['content']
    else:
        from groq import Groq
        import os
        client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
        model  = model or 'llama-3.1-8b-instant'
        resp   = client.chat.completions.create(
            model=model, messages=messages, temperature=temperature
        )
        return resp.choices[0].message.content

print('✓ Función llamar_llm() lista')
print(f'  Usando: {"Ollama" if USE_OLLAMA else "Groq API"}')

In [ ]:
# ── TÉCNICA 1: Zero-shot ──────────────────────────────────────

messages = [
    {"role": "system",
     "content": "Sos un clasificador de sentimientos. Respondés SOLO con: positivo, negativo o neutro."},
    {"role": "user",
     "content": "El sistema funcionó bien al principio pero luego comenzó a fallar constantemente."}
]

print('Zero-shot — sin ejemplos:')
print(f'Prompt: "{messages[-1]["content"]}"')
print()
respuesta = llamar_llm(messages, temperature=0.1)
print(f'Respuesta: {respuesta.strip()}')

In [ ]:
# ── TÉCNICA 2: Few-shot ───────────────────────────────────────

print('\n📖 Brown, T., et al. (2020). Language Models are Few-Shot Learners. https://arxiv.org/abs/2005.14165')

In [ ]:
# ── TÉCNICA 3: Chain of Thought ──────────────────────────────────────────────
# Problema de sistemas: comparación SIN y CON CoT
# El mismo problema — resultado potencialmente diferente

PROBLEMA = (
    "Una base de datos tiene 1.000.000 de usuarios registrados. "
    "El 0,3% son usuarios premium. Cada usuario premium genera "
    "en promedio 15 eventos por día. Los eventos se almacenan en "
    "un log de 512 bytes cada uno. ¿Cuántos GB de logs de eventos "
    "premium se generan por semana?"
)

# Sin CoT
msg_sin_cot = [{"role": "user", "content": PROBLEMA}]

# Con CoT
msg_con_cot = [
    {"role": "system",
     "content": (
         "Sos un ingeniero de sistemas. Resolvé los problemas paso a paso, "
         "mostrando cada operación con su unidad antes de dar la respuesta final."
     )},
    {"role": "user", "content": PROBLEMA}
]

print('PROBLEMA:')
print(PROBLEMA)
print()
print('─' * 65)
print('SIN Chain of Thought:')
resp_sin = llamar_llm(msg_sin_cot, temperature=0.1)
print(resp_sin.strip())

print()
print('─' * 65)
print('CON Chain of Thought:')
resp_con = llamar_llm(msg_con_cot, temperature=0.1)
print(resp_con.strip())

print()
print('📌 Respuesta correcta:')
usuarios_premium = 1_000_000 * 0.003
eventos_dia      = usuarios_premium * 15
eventos_semana   = eventos_dia * 7
bytes_semana     = eventos_semana * 512
gb_semana        = bytes_semana / (1024**3)
print(f'   {usuarios_premium:,.0f} usuarios premium')
print(f'   × 15 eventos/día × 7 días = {eventos_semana:,.0f} eventos/semana')
print(f'   × 512 bytes = {bytes_semana:,.0f} bytes = {gb_semana:.2f} GB')

print()
print('→ CoT reduce errores en razonamiento encadenado')
print('📖 Wei, J., et al. (2022). Chain-of-Thought Prompting. https://arxiv.org/abs/2201.11903')

In [ ]:
# ── TÉCNICA 4: Structured Output ─────────────────────────────

import json

messages = [
    {"role": "system",
     "content": """
Analizás requerimientos de software.
Respondés SOLO con un JSON válido con esta estructura exacta:
{
  "tipo": "funcional" | "no_funcional" | "restriccion",
  "prioridad": "alta" | "media" | "baja",
  "componente": string,
  "resumen": string (max 15 palabras)
}
No incluyas nada más que el JSON."""},
    {"role": "user",
     "content": "El sistema debe responder en menos de 200ms al 95% de las solicitudes bajo carga normal."}
]

print('Structured Output — el modelo devuelve JSON parseable:')
print()
raw = llamar_llm(messages, temperature=0.1)
print('Respuesta raw:')
print(raw.strip())

try:
    # Limpiar posibles bloques de código markdown
    clean = raw.strip().replace('```json', '').replace('```', '').strip()
    parsed = json.loads(clean)
    print()
    print('Parseado correctamente:')
    for k, v in parsed.items():
        print(f'  {k}: {v}')
except json.JSONDecodeError as e:
    print(f'\nError al parsear JSON: {e}')
    print('Tip: ajustar el prompt para reforzar el formato.')

## Context window — el límite que genera RAG

```
  Context window = todo lo que el modelo puede "ver" en una llamada

  ┌─────────────────────────────────────────────────────────┐
  │  System prompt  +  Historial  +  Documento  +  Query   │
  │  (fijo)            (crece)       (grande)     (input)  │
  │                                                         │
  │  ◄─────────────── context window ───────────────────►  │
  │  Claude: 200K tokens   ≈   150.000 palabras            │
  │  Gemini: 1M tokens     ≈   750.000 palabras            │
  │  Llama 4 Scout: 10M    ≈   7.5M palabras               │
  └─────────────────────────────────────────────────────────┘

  Problema real:
  ─────────────
  ✗ Contexto grande = más costo por llamada
  ✗ Contexto grande = más lento
  ✗ "Lost in the middle": el modelo olvida info del medio del contexto
  ✗ Tu empresa tiene 50.000 documentos → no caben en ningún contexto
```

> **Esta limitación es exactamente por qué existe RAG** — lo vemos en Clase 3.

> 📖 *Liu, N. F., et al. (2023). Lost in the Middle: How Language Models Use Long Contexts. TACL 2023. https://arxiv.org/abs/2307.03172*

In [ ]:
# ✏️ EJERCICIO — diseñá tu propio prompt
# Contexto: sos un ingeniero de sistemas que necesita un asistente
# para revisar pull requests de su equipo.
#
# Diseñá un system prompt que:
# 1. Defina el rol del asistente
# 2. Establezca el formato de respuesta (usa structured output)
# 3. Incluya al menos un ejemplo (few-shot)

my_system_prompt = """
# Escribí tu system prompt acá
"""

my_user_message = """
# Escribí el código a revisar acá
def calcular_total(items):
    total = 0
    for i in range(len(items)):
        total = total + items[i]['precio']
    return total
"""

if my_system_prompt.strip() != '# Escribí tu system prompt acá':
    messages = [
        {"role": "system",  "content": my_system_prompt},
        {"role": "user",    "content": my_user_message}
    ]
    print(llamar_llm(messages))
else:
    print('Completá el system prompt para probar tu diseño.')

## Lo que vimos hoy

| Bloque | Concepto clave | Conexión con el curso |
|---|---|---|
| B1 — Tokenizers | BPE y WordPiece en GPT-4o, BERT, Qwen | Cierra Clase 1 |
| B2 — Transformer | Attention resuelve contexto y polisemia | Base de todo LLM |
| B3 — Ciclo de vida | Pre-training → SFT → RLHF/DPO | Por qué instruct ≠ base |
| B4 — Ecosistema | Open weight vs closed, MoE, Kimi K2.5 | Stack del curso |
| B5 — Prompting | Zero-shot, few-shot, CoT, structured output | Base de Clases 3 y 4 |

**Archivos generados:**
- `ecosistema_llm_2026.png`

**Stack que sigue siendo válido:**
- `corpus_embeddings_clase1.npy` + `corpus_clase1.csv` → los usamos en Clase 3

---

## 📚 Bibliografía

### Libros de referencia

- Jurafsky, D. & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). Stanford University.  
  🔗 https://web.stanford.edu/~jurafsky/slp3/

- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.  
  🔗 https://www.deeplearningbook.org/

- Tunstall, L., von Werra, L. & Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly Media.  
  🔗 https://www.oreilly.com/library/view/natural-language-processing/9781098136789/

### Papers

- Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). Attention Is All You Need. *NeurIPS 2017*.  
  🔗 https://arxiv.org/abs/1706.03762

- Devlin, J., Chang, M., Lee, K. & Toutanova, J. (2018). BERT: Pre-training of Deep Bidirectional Transformers. *NAACL 2019*.  
  🔗 https://arxiv.org/abs/1810.04805

- Radford, A., et al. (2018). Improving Language Understanding by Generative Pre-Training. *OpenAI Technical Report*.  
  🔗 https://openai.com/index/language-unsupervised/

- Brown, T., et al. (2020). Language Models are Few-Shot Learners (GPT-3). *NeurIPS 2020*.  
  🔗 https://arxiv.org/abs/2005.14165

- Ouyang, L., et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS 2022*.  
  🔗 https://arxiv.org/abs/2203.02155

- Rafailov, R., et al. (2023). Direct Preference Optimization: Your Language Model is Secretly a Reward Model. *NeurIPS 2023*.  
  🔗 https://arxiv.org/abs/2305.18290

- Wei, J., et al. (2022). Chain-of-Thought Prompting Elicits Reasoning in Large Language Models. *NeurIPS 2022*.  
  🔗 https://arxiv.org/abs/2201.11903

- Shazeer, N., et al. (2017). Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer. *ICLR 2017*.  
  🔗 https://arxiv.org/abs/1701.06538

## Clase 3 — ¿qué viene?

Hoy aprendimos a **hablarle** a un LLM. En Clase 3 vemos cómo darle **conocimiento propio** sin reentrenarlo.

---

- **¿Por qué RAG?** — hallucination, knowledge cutoff, context limits
- **Pipeline RAG naive** — chunking → embeddings → vector store → retrieval → augmentation
- **Hybrid search** — BM25 (Clase 1) + búsqueda semántica (Clase 1) combinados
- **RAG avanzado** — reranking, HyDE, parent-child chunks
- **Práctica** — RAG funcional sobre documentos propios con ChromaDB + Ollama

> Los embeddings del corpus de Clase 1 (`corpus_embeddings_clase1.npy`) los cargamos directamente en el vector store de Clase 3.